In [25]:
from pyaxis import pyaxis
import pandas as pd

In [26]:
# votations_yes_no = r'C:\Users\flora\Desktop\HSLU\Semester 3\Data Visualization\Project\px-x-1703030000_100.px'
votations_results = r'C:\Users\flora\Desktop\HSLU\Semester 3\Data Visualization\Project\data\px-x-1703030000_100.px'
#px_votations_number = pyaxis.parse(votations_yes_no, encoding='ISO-8859-2')
px_votations_results = pyaxis.parse(votations_results, encoding='ISO-8859-2')

#print(px['DATA'])
#print(px['METADATA'])
#%%
#votations_yes_no = px_votations_number['DATA']
#votations_yes_no_meta = px_votations_number['METADATA']

votations_results = px_votations_results['DATA']


In [8]:
votations_results

,Kanton,Datum und Vorlage,Ergebnis,DATA
0,Schweiz,2023-06-18 Änderung des Bundesgesetzes über di...,Stimmberechtigte,5567120
1,Schweiz,2023-06-18 Änderung des Bundesgesetzes über di...,Abgegebene Stimmen,2365154
2,Schweiz,2023-06-18 Änderung des Bundesgesetzes über di...,Beteiligung in %,42.4843366
3,Schweiz,2023-06-18 Änderung des Bundesgesetzes über di...,Gültige Stimmzettel,2321994
4,Schweiz,2023-06-18 Änderung des Bundesgesetzes über di...,Ja,1438216
...,...,...,...,...
128893,Jura,1866-01-14 Festsetzung von Mass und Gewicht,Beteiligung in %,"""..."""
128894,Jura,1866-01-14 Festsetzung von Mass und Gewicht,Gültige Stimmzettel,"""..."""
128895,Jura,1866-01-14 Festsetzung von Mass und Gewicht,Ja,"""..."""
128896,Jura,1866-01-14 Festsetzung von Mass und Gewicht,Nein,"""..."""


### Added Column Datum/Jahr/Monat/Tag hinzugefügt

In [27]:
# Add date column
votations_results[['Datum', 'Vorlage']] = votations_results['Datum und Vorlage'].str.split(' ', n=1, expand=True)
votations_results.drop('Datum und Vorlage', axis=1, inplace=True)

# Add  year, month and day columns
votations_results[['Jahr', 'Monat', 'Tag']] = votations_results['Datum'].str.split('-', n=2, expand=True)

In [18]:

votations_results

,Kanton,Ergebnis,DATA,Datum,Vorlage,Jahr,Monat,Tag
0,Schweiz,Stimmberechtigte,5567120,2023-06-18,Änderung des Bundesgesetzes über die gesetzlic...,2023,06,18
1,Schweiz,Abgegebene Stimmen,2365154,2023-06-18,Änderung des Bundesgesetzes über die gesetzlic...,2023,06,18
2,Schweiz,Beteiligung in %,42.4843366,2023-06-18,Änderung des Bundesgesetzes über die gesetzlic...,2023,06,18
3,Schweiz,Gültige Stimmzettel,2321994,2023-06-18,Änderung des Bundesgesetzes über die gesetzlic...,2023,06,18
4,Schweiz,Ja,1438216,2023-06-18,Änderung des Bundesgesetzes über die gesetzlic...,2023,06,18
...,...,...,...,...,...,...,...,...
128893,Jura,Beteiligung in %,"""...""",1866-01-14,Festsetzung von Mass und Gewicht,1866,01,14
128894,Jura,Gültige Stimmzettel,"""...""",1866-01-14,Festsetzung von Mass und Gewicht,1866,01,14
128895,Jura,Ja,"""...""",1866-01-14,Festsetzung von Mass und Gewicht,1866,01,14
128896,Jura,Nein,"""...""",1866-01-14,Festsetzung von Mass und Gewicht,1866,01,14


## Find numbers for the Bubble Chart

In [33]:
import pandas as pd

topics = ["militär", "ausländ", "atom", "covid"]


filtered_votations = votations_results[votations_results['Kanton'] != 'Schweiz'].drop_duplicates(subset=['Vorlage'])

topic_counts = {topic: filtered_votations['Vorlage'].str.contains(topic, case=False, na=False).sum() for topic in topics}

topic_counts_df = pd.DataFrame(list(topic_counts.items()), columns=['Topic', 'Count'])


In [34]:
topic_counts_df

,Topic,Count
0,militär,18
1,ausländ,15
2,atom,11
3,covid,3


## NLTK used to find the most common words in the initiatives

In [6]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

In [7]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\flora\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\flora\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [word.lower() for word in tokens]
    tokens = [word for word in tokens if word.isalpha() and word not in stopwords.words('german')]
    return tokens

votations_results['Vorlage'] = votations_results['Vorlage'].apply(preprocess_text)

In [12]:
from nltk.probability import FreqDist

all_words = [word for tokens in votations_results['Vorlage'] for word in tokens]
freq = FreqDist(all_words)

print(freq.most_common(100))

[('volksinitiative', 48762), ('bundesbeschluss', 46494), ('bundesgesetz', 22491), ('ťfür', 20979), ('betreffend', 19845), ('bundesverfassung', 13230), ('ť', 7182), ('ťzur', 6615), ('änderung', 6048), ('art', 4725), ('revision', 4536), ('ergänzung', 4536), ('gegenentwurf', 4347), ('massnahmen', 4158), ('schutz', 3969), ('schweiz', 3591), ('schweizerischen', 3402), ('bundesgesetzes', 3213), ('artikel', 3213), ('ťgegen', 3024), ('aufhebung', 2835), ('einführung', 2835), ('artikels', 2835), ('abänderung', 2457), ('weiterführung', 2268), ('neuordnung', 2268), ('bundes', 2079), ('förderung', 1890), ('ahv', 1701), ('erhöhung', 1701), ('ťja', 1701), ('finanzordnung', 1701), ('öffentlichen', 1701), ('rechte', 1701), ('eidgenössischen', 1701), ('aufnahme', 1701), ('bundesgesez', 1701), ('bekämpfung', 1512), ('verbot', 1512), ('finanzierung', 1512), ('armee', 1512), ('verlängerung', 1512), ('ťzum', 1512), ('hinterlassenenversicherung', 1323), ('genehmigung', 1323), ('ausländer', 1323), ('krankenv